### Create Foundational Model
This notebook is intended to help create the foundational model used in our project.

*Note: We trained in on the Mill HPC and it took about an hour*

In [ ]:
#Necessary Modules, requirements.txt included for easy pip install
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.metrics import f1_score, make_scorer
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.base import BaseEstimator, RegressorMixin
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from scipy.stats import uniform, randint
import joblib
import numpy as np
import pandas as pd
import torch
import os
import warnings
ROOT_DIR = '' #add root directory if neccessary 

In [ ]:
main_data = os.path.join(ROOT_DIR, 'traffic_main.csv')
main_df = pd.read_csv(main_data)
warnings.filterwarnings('ignore') #np doesn't like that I was using a copy but it didn't work normally
p_date = ['11','01']
p_j = 1 
p_day = 7
main_df['Time'] = ''
main_df['Day'] = ''
main_df['Month'] = ''
main_df['Date'] = ''
df = main_df.copy()
for index, t in main_df.iterrows():
    if (p_j != t['Junction']):
        p_j += 1
        p_day = 6
        p_date = ['11','01']
    date_time = t['DateTime'].split()
    date = date_time[0].split('-')
    final_string = 0 
    if(int(date[2]) != int(p_date[1])):  
        match p_day:
            case 6:
                final_string = 7
            case 7:
                final_string = 1
            case 1:
                final_string = 2
            case 2:
                final_string = 3
            case 3:
                final_string = 4
            case 4:
                final_string = 5
            case 5:
                final_string = 6
        p_day = final_string 
    else:
        final_string = p_day
    p_date = [date[1], date[2]] 
    df['Day'][index] = final_string
    df['Time'][index] = date_time[1]
    date_time = t['DateTime'].split()
    date = date_time[0].split('-')
    df['Date'][index] = int(date[2]) 
    df['Month'][index] = int(date[1])

In [ ]:
#run if you have the precreated csv
main_df = pd.read_csv(os.path.join(ROOT_DIR, 'pre_processed.csv'))

In [ ]:
main_df["dow_sin"] = np.sin(2 * np.pi * (main_df["Day"]-1) / 7)
main_df["dow_cos"] = np.cos(2 * np.pi * (main_df["Day"]-1) / 7)
main_df["is_weekend"] = main_df["Day"] >= 5
main_df["is_business_hours"] = main_df["Time"].between(9,17)
main_df["month_sin"] = np.sin(2*np.pi*main_df["Month"]/12)
main_df["month_cos"] = np.cos(2*np.pi*main_df["Month"]/12)

In [ ]:
class CatBoostWrapper(BaseEstimator, RegressorMixin):

    def __init__(self, **params):
        self.params = params

    def fit(self, X, y):
        self.model_ = CatBoostRegressor(
            **self.params,
            allow_writing_files=False,
            verbose=0
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def get_params(self, deep=True):
        return self.params.copy()

    def set_params(self, **params):
        self.params.update(params)
        return self

class junctionEnsemble():
    def __init__(self, df, name):
        split_index = int(len(df) * 0.8)
        train_df = df.iloc[:split_index]
        test_df  = df.iloc[split_index:]

        self.X_train = train_df.drop(columns=["Vehicles","ID","DateTime", "Junction"])
        self.y_train = train_df["Vehicles"]

        self.X_test  = test_df.drop(columns=["Vehicles"])
        self.y_test  = test_df["Vehicles"]
        self.name = name
        self.best_lgbm = None
        self.best_xgb = None
        self.best_cat = None
        self.ensemble_model = None

    #consider adding SVM or Linear model to the final retrained model 
    def train(self):
        param_grid_xgb = {
            "n_estimators": randint(100, 1000),
            "max_depth": randint(3, 10),
            "min_child_weight": randint(1, 10),
            "learning_rate": uniform(0.01, 0.3),        
            "gamma": uniform(0, 0.5),
            "subsample": uniform(0.5, 0.5),              
            "colsample_bytree": uniform(0.5, 0.5),       
            "colsample_bylevel": uniform(0.5, 0.5),
            "reg_alpha": uniform(0, 1),                  
            "reg_lambda": uniform(0.5, 2),
        }
        param_grid_lm = {
            "n_estimators": randint(100, 1000),
            "max_depth": randint(3, 12),
            "num_leaves": randint(20, 150),               
            "learning_rate": uniform(0.01, 0.3),
            "min_child_samples": randint(5, 100),         
            "subsample": uniform(0.5, 0.5),
            "subsample_freq": randint(1, 10),             
            "colsample_bytree": uniform(0.5, 0.5),
            "reg_alpha": uniform(0, 1),                   
            "reg_lambda": uniform(0, 2),
        }
        param_grid_cat = {
            "iterations": randint(100, 1000),
            "depth": randint(3, 10),
            "learning_rate": uniform(0.01, 0.3),
            "l2_leaf_reg": uniform(1, 10),
            "min_data_in_leaf": randint(1, 50),
            "colsample_bylevel": uniform(0.5, 0.5),
            "random_strength": uniform(0, 2),
        }

        cat_model = CatBoostRegressor(
            verbose=0,
            random_state=42,
            allow_writing_files=False
        )

        grid_search_cat = RandomizedSearchCV(
            estimator=cat_model,
            param_distributions=param_grid_cat,
            scoring="neg_root_mean_squared_error",
            cv=5,
            verbose=0,
            n_jobs=-1,
            n_iter=100,
            refit=False  
        )

        grid_search_cat.fit(self.X_train, self.y_train)

        self.best_cat = CatBoostWrapper(
            **grid_search_cat.best_params_,
            random_state=42
        )

        
        lgbm_model = LGBMRegressor(random_state=42, n_jobs=-1)
        grid_search_lm = RandomizedSearchCV(
            estimator=lgbm_model,
            param_distributions=param_grid_lm,
            scoring="neg_root_mean_squared_error",
            cv=5,
            verbose=0,
            n_jobs=-1
        )
        grid_search_lm.fit(self.X_train, self.y_train)
        self.best_lgbm = grid_search_lm.best_estimator_

        xgb_model = XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=42, n_jobs=-1)
        grid_search_xgb = RandomizedSearchCV(
            estimator = xgb_model,
            param_distributions=param_grid_xgb,
            scoring="neg_root_mean_squared_error",
            n_iter=100,
            cv=5,
            verbose=0,
            n_jobs=-1
        )
        grid_search_xgb.fit(self.X_train, self.y_train)
        self.best_xgb = grid_search_xgb.best_estimator_

        
        stack = StackingRegressor(
            estimators=[
                ("lgb", self.best_lgbm),
                ("xgb", self.best_xgb),
                ("cat", self.best_cat)
            ],
            final_estimator=Ridge(),
            passthrough=True
        )
        stack.fit(self.X_train, self.y_train)
        self.ensemble_method = stack

    def save(self, file_path):
        joblib.dump(self.ensemble_method, file_path)

In [ ]:
junc_models = []
for j in main_df["Junction"].unique():
    temp_df = main_df[main_df["Junction"] == j]
    model = junctionEnsemble(temp_df, j)
    model.train()
    model.save(f'{j}-model.pkl')
    junc_models.append(model)